# Задание 3. Подсчёт автомобилей и оценка загруженности парковки по видео с камеры наблюдения с использованием YOLOv8 и трекинга ByteTrack

| | |
|---|---|
| **Входные данные** | видеозапись парковки с камеры наблюдения: минутный фрагмент записи [BLK-HDPTZ12 Security Camera Parking Lot Surveillance Video](https://commons.wikimedia.org/wiki/File:BLK-HDPTZ12_Security_Camera_Parking_Lot_Surveillance_Video.webm) (Wikimedia Commons, общественное достояние), 1280×720, 15 к/с |
| **Что определяет система** | число автомобилей в кадре и в зоне парковки, число уникальных машин за запись, загруженность в % от вместимости |
| **Модель** | YOLOv8n, предобученная на COCO (классы car, motorcycle, bus, truck), без дообучения; трекинг объектов ByteTrack |
| **Оценка** | сравнение с ручным подсчётом на 25 контрольных кадрах (разметка лежит в `ground_truth/parking.csv`): MAE, MAPE, доля кадров с ошибкой ≤ 1 машины |

Ноутбук работает и в Google Colab, и локально, и выполняется целиком командой **«Выполнить все»** (Run all): видео и ручная разметка уже лежат в репозитории, зависимости ставятся сами. В Colab можно включить GPU (**Среда выполнения → Сменить среду выполнения → T4 GPU**), но и на процессоре всё отработает за несколько минут.

## 1. Подготовка

In [ ]:
import sys, os, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("bigdata-practice").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/farzXr/bigdata-practice.git"], check=True)
    os.chdir("bigdata-practice/task3_parking_yolo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "lap"], check=True)
else:
    if not Path("count_cars.py").exists() and Path("task3_parking_yolo").exists():
        os.chdir("task3_parking_yolo")  # ноутбук открыт из корня репозитория
    if importlib.util.find_spec("ultralytics") is None or importlib.util.find_spec("lap") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from count_cars import pick_device
print("Рабочая папка:", Path.cwd())
print("Устройство:", pick_device())

## 2. Видео

Минутный фрагмент записи камеры наблюдения на парковке лежит в репозитории: `videos/parking.mp4`.
Если файла нет (например, ноутбук скачан отдельно от репозитория), ячейка ниже загрузит его с GitHub.

Чтобы проверить систему на своём видео, положите `.mp4` в `videos/` и укажите его в `VIDEO`;
тогда задайте свои `ROI` и `CAPACITY` в разделе 3 и разметку в разделе 5.

In [ ]:
import urllib.request

VIDEO_URL = "https://raw.githubusercontent.com/farzXr/bigdata-practice/main/task3_parking_yolo/videos/parking.mp4"
VIDEO = Path("videos/parking.mp4")

if not VIDEO.exists():
    VIDEO.parent.mkdir(exist_ok=True)
    print("Скачиваю видео...")
    urllib.request.urlretrieve(VIDEO_URL, VIDEO)
print("Видео:", VIDEO, f"({VIDEO.stat().st_size / 1e6:.1f} МБ)")

In [ ]:
# Первый кадр: по нему удобно выбрать зону парковки (ROI) в долях кадра
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(str(VIDEO))
ok, frame = cap.read()
cap.release()
h, w = frame.shape[:2]
plt.figure(figsize=(11, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), extent=[0, 1, 1, 0])
plt.xticks([i / 10 for i in range(11)]); plt.yticks([i / 10 for i in range(11)])
plt.grid(color="yellow", alpha=0.5)
plt.title(f"Первый кадр {w}x{h}: координаты в долях для параметра ROI")
plt.show()

## 3. Подсчёт машин

Параметры:
- `ROI` — зона парковки в долях кадра (x,y через `;`, по часовой стрелке); `None` — считать во всём кадре.
  Для демонстрационного видео это два ближних ряда: нижние 40 % кадра без обрезанных краёв.
  Машина считается в зоне, если в неё попадает центр её рамки;
- `CAPACITY` — число мест в зоне (12: 7 в первом ряду и 5 в нижнем), чтобы посчитать загруженность в %;
- `STRIDE` — обрабатывать каждый N-й кадр: 1 — все, 2 — вдвое быстрее (используется на процессоре).

In [ ]:
ROI = "0.07,0.6;0.96,0.6;0.96,1;0.07,1"   # для своего видео подберите по сетке на первом кадре
CAPACITY = 12
STRIDE = 1 if pick_device() != "cpu" else 2

cmd = [sys.executable, "count_cars.py", "--source", str(VIDEO), "--stride", str(STRIDE)]
if ROI:
    cmd += ["--roi", ROI]
if CAPACITY:
    cmd += ["--capacity", str(CAPACITY)]
subprocess.run(cmd, check=True)
OUT = Path("out") / VIDEO.stem

## 4. Результаты

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

summary = json.loads((OUT / "summary.json").read_text(encoding="utf-8"))
display(pd.Series(summary, name="значение").to_frame())
display(Image(str(OUT / "occupancy.png")))

In [ ]:
# Несколько кадров из размеченного видео
cap = cv2.VideoCapture(str(OUT / "annotated.mp4"))
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, idx in zip(axes, [0, n // 2, max(n - 1, 0)]):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, img = cap.read()
    if ok:
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"кадр {idx * STRIDE}")
    ax.axis("off")
cap.release()
plt.tight_layout()
plt.show()

counts = pd.read_csv(OUT / "counts.csv")
counts.describe().round(2)

## 5. Оценка точности

На 25 контрольных кадрах машины в зоне посчитаны вручную по тому же правилу, что и у системы (центр машины внутри зоны);
разметка лежит в `ground_truth/<имя видео>.csv`. Номера кадров чётные, поэтому разметка подходит и для `STRIDE = 1`, и для `STRIDE = 2`.

Ячейка ниже выгружает контрольные кадры в `out/<видео>/frames/` (их можно открыть и проверить разметку) и подставляет готовую разметку.
Для своего видео разметки нет: ячейка создаст пустой `ground_truth.csv`, посчитайте машины на кадрах, впишите числа в колонку `true_count`
и запустите следующую ячейку ещё раз.

In [ ]:
import shutil

GT_STRIDE = 2  # кадры разметки кратны 2 и есть в результатах и при STRIDE = 1, и при STRIDE = 2
bundled = Path("ground_truth") / f"{VIDEO.stem}.csv"
if bundled.exists():
    shutil.copy(bundled, OUT / "ground_truth.csv")
    print("Готовая разметка:", bundled)

subprocess.run([sys.executable, "evaluate.py", "extract", "--source", str(VIDEO),
                "--n", "25", "--stride", str(GT_STRIDE)], check=True)
if bundled.exists():  # на случай, если extract пересоздал таблицу под другой набор кадров
    shutil.copy(bundled, OUT / "ground_truth.csv")

In [ ]:
gt = pd.read_csv(OUT / "ground_truth.csv")
if gt["true_count"].isna().all():
    print("Сначала заполните true_count в", OUT / "ground_truth.csv")
    metrics = None
else:
    subprocess.run([sys.executable, "evaluate.py", "score", "--source", str(VIDEO)], check=True)
    metrics = json.loads((OUT / "evaluation.json").read_text(encoding="utf-8"))
    display(Image(str(OUT / "evaluation.png")))

    # контрольные кадры: ручной подсчёт против системы
    check = gt.merge(counts[["frame", "in_roi"]], on="frame").rename(
        columns={"frame": "Кадр", "true_count": "Вручную", "in_roi": "YOLOv8"})
    with pd.option_context("display.max_columns", None, "display.width", 250):
        display(check.set_index("Кадр").T)

## 6. Выводы

In [ ]:
# Итоговые цифры подставляются автоматически из результатов выше
print(f"1. Обработано {summary['frames_processed']} кадров со скоростью {summary['processing_fps']} к/с "
      f"на устройстве {summary['device']}.")
print(f"2. Машин в зоне парковки: в среднем {summary['cars_mean']}, от {summary['cars_min']} до {summary['cars_max']}; "
      + (f"средняя загруженность {summary['occupancy_mean']:.0%} из {summary['capacity']} мест." if summary["capacity"] else "вместимость не задана."))
print(f"3. Трекер насчитал {summary['unique_vehicles_tracked']} разных машин в зоне.")
if metrics:
    print(f"4. Сравнение с ручным подсчётом на {metrics['frames']} кадрах: MAE = {metrics['MAE']} машины, "
          f"MAPE = {metrics['MAPE_%']}%, точное совпадение в {metrics['exact_%']}% кадров, "
          f"ошибка не больше одной машины в {metrics['within_1_%']}% кадров.")

**Интерпретация.**

- Предобученная YOLOv8n распознаёт легковые машины без дообучения: класс `car` есть в COCO, а модель весит около 6 МБ и работает в реальном времени даже на ноутбуке.
- В зоне двух ближних рядов машины крупные и почти не перекрываются, поэтому число машин в кадре система определяет практически без ошибок — в том числе в моменты, когда машина выезжает с места и проезжает по проезду.
- Число уникальных машин по ID трекера получается немного больше реального (в зоне 9 стоящих машин и одна проезжающая): когда машина частично закрыта или выезжает с места, ByteTrack иногда теряет её и выдаёт новый ID. Поэтому для загруженности надёжнее покадровый подсчёт, а число уникальных ID — оценка сверху.
- Сглаживание медианой за 1 секунду убирает мерцание: на отдельных кадрах детектор может пропустить машину, частично закрытую другой.
- На дальних рядах машины мелкие и сильно перекрываются — там ошибок больше. Улучшить результат помогают больший `--imgsz` (1280), более тяжёлая модель (`yolov8s.pt`) или ROI, отсекающий дальний план, как сделано здесь.
- Применение: подсказки о свободных местах, статистика загрузки по часам, планирование парковочного пространства.